# 🧠 Techniques Avancées de Deep Learning

Ce notebook couvre **5 exercices avancés** couvrant les grands défis du deep learning moderne :

| # | Exercice | Technique clé |
|---|----------|---------------|
| 1 | Optimisation avancée | LR Scheduling + Gradient Clipping |
| 2 | Modèle génératif | GAN (DCGAN sur MNIST) |
| 3 | Reinforcement Learning | DQN sur CartPole |
| 4 | Données déséquilibrées | SMOTE + Focal Loss |
| 5 | Robustesse adversariale | FGSM + Adversarial Training |

> ⚠️ **GPU recommandé** : `Runtime > Change runtime type > Hardware accelerator > GPU (T4)`

In [ ]:
# ── Installation des dépendances ──────────────────────────────────────────────
!pip install -q torch torchvision gymnasium imbalanced-learn matplotlib seaborn scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, SGD

sns.set_theme(style='darkgrid')
%matplotlib inline

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Environnement prêt — Appareil : {DEVICE}")

---
# 🏋️ Exercice 1 — Optimisation Avancée

## Objectif
Améliorer la **stabilité** et la **convergence** de l'entraînement grâce à deux techniques :

- **Learning Rate Scheduling (Cosine Annealing)** : fait varier le taux d'apprentissage selon une courbe cosinus, évitant de rester coincé dans des minima locaux
- **Gradient Clipping** : plafonne la norme des gradients pour éviter les *exploding gradients* (instabilité numérique)

On utilisera un **CNN simple sur CIFAR-10** et on comparera avec/sans ces techniques.

In [ ]:
# ── Chargement CIFAR-10 ───────────────────────────────────────────────────────
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalisation RGB
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader_cifar = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader_cifar  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=2)

CLASSES = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')
print(f"✅ CIFAR-10 chargé : {len(train_set)} train | {len(test_set)} test")

In [ ]:
# ── Architecture CNN ──────────────────────────────────────────────────────────
class SimpleCNN(nn.Module):
    """
    CNN léger pour CIFAR-10.
    Architecture : Conv → BN → ReLU → MaxPool (×2) → FC → Sortie
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def train_with_config(use_scheduler: bool, use_clip: bool, epochs: int = 15):
    """
    Entraîne le CNN avec ou sans LR scheduling / gradient clipping.
    Retourne les listes de loss et d'accuracy par epoch.
    """
    model = SimpleCNN().to(DEVICE)
    optimizer = SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss()

    # Cosine Annealing : le LR suit une courbe cosinus de 0.1 → ~0
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs) \
                if use_scheduler else None

    train_losses, train_accs = [], []

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for X_batch, y_batch in train_loader_cifar:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()

            # Gradient clipping : plafonne la norme L2 des gradients à 1.0
            if use_clip:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(y_batch).sum().item()
            total   += y_batch.size(0)

        if scheduler:
            scheduler.step()

        train_losses.append(running_loss / len(train_loader_cifar))
        train_accs.append(100. * correct / total)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"  Epoch {epoch+1:>2}/{epochs} | Loss: {train_losses[-1]:.3f} | "
              f"Acc: {train_accs[-1]:.1f}% | LR: {current_lr:.5f}")

    return train_losses, train_accs


print("🔵 Entraînement SANS optimisation avancée...")
losses_base, accs_base = train_with_config(use_scheduler=False, use_clip=False)

print("\n🟢 Entraînement AVEC LR Scheduling + Gradient Clipping...")
losses_adv, accs_adv = train_with_config(use_scheduler=True, use_clip=True)

In [ ]:
# ── Comparaison visuelle ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(losses_base) + 1)

axes[0].plot(epochs_range, losses_base, label='Sans optimisation', color='tomato',    linewidth=2)
axes[0].plot(epochs_range, losses_adv,  label='Avec LR sched + clip', color='royalblue', linewidth=2)
axes[0].set_title('Loss par epoch', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()

axes[1].plot(epochs_range, accs_base, label='Sans optimisation', color='tomato',    linewidth=2)
axes[1].plot(epochs_range, accs_adv,  label='Avec LR sched + clip', color='royalblue', linewidth=2)
axes[1].set_title('Accuracy par epoch', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.suptitle('Exercice 1 — Impact des techniques d\'optimisation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 📝 Analyse — Techniques d'optimisation avancées

**Le Cosine Annealing** fait décroître le learning rate selon une courbe cosinus, permettant des grands pas en début d'entraînement (exploration) puis des petits pas en fin (convergence fine). Cela évite d'osciller autour d'un minimum sans jamais y converger.

**Le Gradient Clipping** empêche les gradients de devenir trop grands lors de la rétropropagation, un phénomène appelé *exploding gradients* particulièrement fréquent avec les RNN et les réseaux profonds. En plafonnant la norme à 1.0, on garantit des mises à jour de poids stables.

Les courbes montrent que le modèle avec ces deux techniques converge **plus rapidement et plus régulièrement** : la loss descend plus vite et l'accuracy est plus stable d'une epoch à l'autre, sans les pics d'instabilité visibles dans le modèle de base.

Ces techniques sont devenues des **bonnes pratiques universelles** du deep learning et sont recommandées pour tout entraînement, en particulier sur des architectures profondes ou des données bruitées.

---
# 🎨 Exercice 2 — Modèle Génératif (DCGAN sur MNIST)

## Objectif
Construire un **GAN (Generative Adversarial Network)** pour générer des chiffres manuscrits réalistes.

### Comment fonctionne un GAN ?
Un GAN est composé de **deux réseaux en compétition** :
- **Générateur (G)** : prend du bruit aléatoire → génère des fausses images
- **Discriminateur (D)** : doit distinguer les vraies images des fausses

Ils s'entraînent ensemble dans un jeu adversarial : G essaie de tromper D, D essaie de ne pas être trompé. À l'équilibre, G génère des images indiscernables des vraies.

```
Bruit z ──► Générateur ──► Fausse image ──►┐
                                             ├──► Discriminateur ──► Vrai / Faux
Vraie image ─────────────────────────────────┘
```

In [ ]:
# ── Dataset MNIST ─────────────────────────────────────────────────────────────
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # Images en [-1, 1] (comme la sortie tanh du générateur)
])

mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=mnist_transform)
mnist_loader = DataLoader(mnist_train, batch_size=128, shuffle=True, num_workers=2)

print(f"✅ MNIST chargé : {len(mnist_train)} images (28×28 px, niveaux de gris)")

In [ ]:
# ── Architecture DCGAN ────────────────────────────────────────────────────────
LATENT_DIM = 100  # Dimension du vecteur de bruit en entrée du générateur

class Generator(nn.Module):
    """
    Générateur DCGAN.
    Transforme un vecteur de bruit (100,) en une image 28×28.
    Utilise des ConvTranspose2d (déconvolutions) pour upsampler.
    Activation finale : Tanh → sortie dans [-1, 1]
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # Bruit → feature maps 7×7
            nn.Linear(LATENT_DIM, 256 * 7 * 7),
            nn.Unflatten(1, (256, 7, 7)),
            nn.BatchNorm2d(256), nn.ReLU(True),
            # 7×7 → 14×14
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(True),
            # 14×14 → 28×28
            nn.ConvTranspose2d(128, 1, 4, stride=2, padding=1),
            nn.Tanh()  # Sortie dans [-1, 1] comme les vraies images normalisées
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    """
    Discriminateur DCGAN.
    Reçoit une image 28×28 et prédit : vrai (1) ou faux (0).
    Activation finale : Sigmoid → probabilité
    Utilise LeakyReLU (recommandé pour les discriminateurs GAN)
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # 28×28 → 14×14
            nn.Conv2d(1, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            # 14×14 → 7×7
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            # Classification
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)

print(f"✅ Générateur     : {sum(p.numel() for p in G.parameters()):,} paramètres")
print(f"✅ Discriminateur : {sum(p.numel() for p in D.parameters()):,} paramètres")

In [ ]:
# ── Entraînement du GAN ───────────────────────────────────────────────────────
# Les deux réseaux ont leurs propres optimiseurs
opt_G = Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))  # betas recommandés pour GAN
opt_D = Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion_gan = nn.BCELoss()  # Binary Cross-Entropy pour Vrai/Faux

GAN_EPOCHS = 20
g_losses, d_losses = [], []

# Vecteur de bruit fixe pour visualiser la progression du générateur
fixed_noise = torch.randn(16, LATENT_DIM).to(DEVICE)

print(f"🎨 Entraînement GAN sur {GAN_EPOCHS} epochs...")

for epoch in range(GAN_EPOCHS):
    g_loss_epoch, d_loss_epoch = 0.0, 0.0

    for real_imgs, _ in mnist_loader:
        real_imgs = real_imgs.to(DEVICE)
        batch_sz  = real_imgs.size(0)

        real_labels = torch.ones(batch_sz, 1).to(DEVICE)
        fake_labels = torch.zeros(batch_sz, 1).to(DEVICE)

        # ── Entraîner le Discriminateur ───────────────────────────────────────
        # Objectif D : reconnaître les vraies (→1) et les fausses (→0)
        opt_D.zero_grad()
        loss_real = criterion_gan(D(real_imgs), real_labels)
        z = torch.randn(batch_sz, LATENT_DIM).to(DEVICE)
        fake_imgs = G(z).detach()  # .detach() pour ne pas propager dans G
        loss_fake = criterion_gan(D(fake_imgs), fake_labels)
        loss_D = (loss_real + loss_fake) / 2
        loss_D.backward()
        opt_D.step()

        # ── Entraîner le Générateur ───────────────────────────────────────────
        # Objectif G : tromper D → faire croire que ses images sont vraies (→1)
        opt_G.zero_grad()
        z = torch.randn(batch_sz, LATENT_DIM).to(DEVICE)
        fake_imgs = G(z)
        loss_G = criterion_gan(D(fake_imgs), real_labels)  # On veut que D dise "vrai"
        loss_G.backward()
        opt_G.step()

        g_loss_epoch += loss_G.item()
        d_loss_epoch += loss_D.item()

    g_losses.append(g_loss_epoch / len(mnist_loader))
    d_losses.append(d_loss_epoch / len(mnist_loader))
    print(f"  Epoch {epoch+1:>2}/{GAN_EPOCHS} | Loss_G: {g_losses[-1]:.4f} | Loss_D: {d_losses[-1]:.4f}")

print("\n✅ Entraînement GAN terminé !")

In [ ]:
# ── Visualisation des images générées ────────────────────────────────────────
G.eval()
with torch.no_grad():
    generated = G(fixed_noise).cpu()

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    img = generated[i, 0].numpy()
    ax.imshow(img, cmap='gray', vmin=-1, vmax=1)
    ax.axis('off')
plt.suptitle('Images générées par le GAN (après entraînement)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Courbe des losses ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(g_losses, label='Loss Générateur',     color='royalblue', linewidth=2)
ax.plot(d_losses, label='Loss Discriminateur', color='tomato',    linewidth=2)
ax.set_title('Évolution des losses GAN', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
ax.legend()
plt.tight_layout()
plt.show()

print("💡 Un bon GAN converge vers Loss_G ≈ Loss_D ≈ 0.693 (log(2) = équilibre de Nash)")

### 📝 Réflexion — Modèles génératifs

L'entraînement d'un GAN est intrinsèquement **instable** : si le discriminateur devient trop fort trop vite, le générateur ne reçoit plus de signal utile (gradient vanishing) ; à l'inverse, si le générateur domine, le discriminateur ne discrimine plus. Maintenir cet équilibre est l'un des défis majeurs du domaine.

Le **DCGAN** (Deep Convolutional GAN) apporte des stabilisateurs clés : BatchNorm, LeakyReLU dans le discriminateur, et des betas spécifiques pour Adam (0.5, 0.999), qui ont été empiriquement démontrés plus stables que les valeurs par défaut.

Les applications des modèles génératifs sont immenses : génération d'images synthétiques pour **augmenter des datasets médicaux** rares, création de contenus artistiques, synthèse de voix, simulation de scénarios pour le reinforcement learning, et anonymisation de données sensibles.

---
# 🕹️ Exercice 3 — Reinforcement Learning (DQN sur CartPole)

## Objectif
Entraîner un **agent DQN** à maintenir un poteau en équilibre sur un chariot.

### Rappel : Comment fonctionne le RL ?
```
Agent ──(action)──► Environnement ──(état, récompense)──► Agent
```
L'agent apprend une **politique** (quelle action choisir dans chaque état) pour **maximiser la récompense cumulée**.

### DQN (Deep Q-Network)
Le DQN utilise un réseau de neurones pour approximer la fonction **Q(état, action)** qui estime la récompense future espérée. Il introduit deux innovations clés :
- **Experience Replay** : mémorise les transitions passées et les rejoue aléatoirement → casse les corrélations temporelles
- **Target Network** : réseau cible figé mis à jour périodiquement → stabilise l'entraînement

In [ ]:
import gymnasium as gym
from collections import deque, namedtuple
import random

# ── Environnement CartPole ────────────────────────────────────────────────────
# État  : [position chariot, vitesse chariot, angle poteau, vitesse angulaire]
# Action: 0 (pousser à gauche) ou 1 (pousser à droite)
# Récompense : +1 à chaque pas de temps où le poteau est debout
# Épisode terminé si : angle > 12° ou position > 2.4m ou 500 steps

env = gym.make('CartPole-v1')
STATE_DIM  = env.observation_space.shape[0]   # 4
ACTION_DIM = env.action_space.n               # 2

print(f"✅ Environnement CartPole-v1 créé")
print(f"   Espace d'état  : {STATE_DIM} dimensions")
print(f"   Espace d'action: {ACTION_DIM} actions")

In [ ]:
# ── Architecture du réseau Q ──────────────────────────────────────────────────
class QNetwork(nn.Module):
    """
    Réseau de neurones qui prédit Q(s, a) pour toutes les actions.
    Entrée  : état (4 dimensions)
    Sortie  : Q-values pour chaque action (2 valeurs)
    """
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128), nn.ReLU(),
            nn.Linear(128, 128),       nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, x):
        return self.net(x)


# ── Replay Buffer ─────────────────────────────────────────────────────────────
Transition = namedtuple('Transition', ['state', 'action', 'reward', 'next_state', 'done'])

class ReplayBuffer:
    """
    Mémoire circulaire qui stocke les transitions (s, a, r, s', done).
    Lors de l'entraînement, on tire un mini-batch aléatoire pour briser les corrélations.
    """
    def __init__(self, capacity=10_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)


# ── Agent DQN ─────────────────────────────────────────────────────────────────
class DQNAgent:
    """
    Agent DQN avec :
    - Réseau Q (online) + réseau cible (target) mis à jour périodiquement
    - Politique ε-greedy : exploration décroissante au fil du temps
    - Experience Replay
    """
    def __init__(self):
        self.q_net     = QNetwork(STATE_DIM, ACTION_DIM).to(DEVICE)
        self.target_net= QNetwork(STATE_DIM, ACTION_DIM).to(DEVICE)
        self.target_net.load_state_dict(self.q_net.state_dict())  # Synchronisation initiale
        self.target_net.eval()

        self.optimizer = Adam(self.q_net.parameters(), lr=1e-3)
        self.buffer    = ReplayBuffer()
        self.criterion = nn.MSELoss()

        # Hyperparamètres ε-greedy
        self.epsilon     = 1.0    # 100% exploration au début
        self.eps_min     = 0.01
        self.eps_decay   = 0.995
        self.gamma       = 0.99   # Facteur d'actualisation (importance du futur)
        self.batch_size  = 64
        self.target_freq = 10     # Mise à jour du réseau cible toutes les 10 epochs

    def select_action(self, state):
        """Politique ε-greedy : exploration (aléatoire) ou exploitation (Q-max)."""
        if random.random() < self.epsilon:
            return random.randint(0, ACTION_DIM - 1)  # Exploration
        state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            return self.q_net(state_t).argmax().item()  # Exploitation

    def train_step(self):
        """Met à jour le réseau Q sur un mini-batch du replay buffer."""
        if len(self.buffer) < self.batch_size:
            return None

        batch = self.buffer.sample(self.batch_size)
        states      = torch.FloatTensor([t.state      for t in batch]).to(DEVICE)
        actions     = torch.LongTensor( [t.action     for t in batch]).to(DEVICE)
        rewards     = torch.FloatTensor([t.reward     for t in batch]).to(DEVICE)
        next_states = torch.FloatTensor([t.next_state for t in batch]).to(DEVICE)
        dones       = torch.FloatTensor([t.done       for t in batch]).to(DEVICE)

        # Q-values courantes pour les actions choisies
        q_values = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze()

        # Q-values cibles (équation de Bellman)
        # Q_target = r + γ * max_a' Q_target(s', a')  (si épisode pas terminé)
        with torch.no_grad():
            next_q = self.target_net(next_states).max(1)[0]
            q_targets = rewards + self.gamma * next_q * (1 - dones)

        loss = self.criterion(q_values, q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.optimizer.step()
        return loss.item()

    def update_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)

    def sync_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())


print("✅ Agent DQN défini.")

In [ ]:
# ── Entraînement de l'agent ───────────────────────────────────────────────────
agent = DQNAgent()
NUM_EPISODES = 300
episode_rewards = []

print(f"🕹️  Entraînement sur {NUM_EPISODES} épisodes...")

for episode in range(NUM_EPISODES):
    state, _ = env.reset()
    total_reward = 0

    for step in range(500):  # Max 500 steps par épisode
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        agent.buffer.push(state, action, reward, next_state, float(done))
        agent.train_step()

        state = next_state
        total_reward += reward
        if done:
            break

    agent.update_epsilon()
    if episode % agent.target_freq == 0:
        agent.sync_target()

    episode_rewards.append(total_reward)
    if (episode + 1) % 50 == 0:
        avg = np.mean(episode_rewards[-50:])
        print(f"  Episode {episode+1:>4} | Récompense moy (50 eps): {avg:.1f} | ε: {agent.epsilon:.3f}")

print(f"\n✅ Entraînement terminé ! Récompense finale (moy 50 eps) : {np.mean(episode_rewards[-50:]):.1f}/500")

In [ ]:
# ── Courbe d'apprentissage de l'agent ─────────────────────────────────────────
window = 20  # Moyenne glissante sur 20 épisodes
smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(episode_rewards, alpha=0.3, color='steelblue', label='Récompense brute')
ax.plot(range(window-1, len(episode_rewards)), smoothed, color='royalblue', linewidth=2, label=f'Moyenne glissante ({window} eps)')
ax.axhline(500, color='green', linestyle='--', linewidth=1.5, label='Score parfait (500)')
ax.set_title('Exercice 3 — Progression de l\'agent DQN sur CartPole', fontsize=13)
ax.set_xlabel('Épisode'); ax.set_ylabel('Récompense totale')
ax.legend()
plt.tight_layout()
plt.show()

### 📝 Résumé — Reinforcement Learning

L'entraînement RL est fondamentalement différent du supervised learning : il n'y a pas de vérité terrain, juste un signal de récompense **sparse et retardé**. L'agent doit explorer suffisamment pour découvrir de bonnes stratégies, puis exploiter ses connaissances — le dilemme **exploration/exploitation** est au cœur du RL.

Le **DQN** résout les problèmes de corrélation temporelle et d'instabilité des Q-réseaux grâce au **replay buffer** (qui mélange les expériences passées) et au **réseau cible** (qui fournit des cibles stables). Sans ces innovations, l'entraînement diverge presque systématiquement.

Les applications réelles du RL sont transformationnelles : AlphaGo/AlphaZero ont battu les meilleurs joueurs mondiaux aux échecs et au Go, des agents RL gèrent le refroidissement de datacenters Google (économies d'énergie de 40%), et les systèmes de recommandation utilisent le RL pour maximiser l'engagement sur de longues séquences d'interactions.

---
# ⚖️ Exercice 4 — Données Déséquilibrées

## Objectif
Améliorer la performance d'un modèle sur un dataset **très déséquilibré** (fraude bancaire : ~0.17% de fraudes).

### Techniques abordées
| Technique | Type | Description |
|-----------|------|-------------|
| **SMOTE** | Sur-échantillonnage | Génère des exemples synthétiques de la classe minoritaire |
| **Focal Loss** | Fonction de perte | Pénalise plus les exemples difficiles (mal classés) |
| **Weighted CE** | Fonction de perte | Attribue un poids plus élevé à la classe minoritaire |

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from imblearn.over_sampling import SMOTE
import torch

# ── Génération d'un dataset déséquilibré (simulation fraude bancaire) ─────────
# weights=[0.995, 0.005] → 0.5% de fraudes (très déséquilibré)
X_raw, y_raw = make_classification(
    n_samples=10_000,
    n_features=20,
    n_informative=10,
    weights=[0.98, 0.02],   # 98% classe 0, 2% classe 1 (fraude)
    random_state=42
)

print("📊 Distribution originale :")
unique, counts = np.unique(y_raw, return_counts=True)
for u, c in zip(unique, counts):
    print(f"   Classe {u}: {c} ({c/len(y_raw)*100:.1f}%)")

# Découpage train / test
X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)

# Visualisation du déséquilibre
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Normal (0)', 'Fraude (1)'], np.bincount(y_raw), color=['steelblue', 'tomato'])
ax.set_title('Déséquilibre des classes', fontsize=12)
ax.set_ylabel('Nombre d\'exemples')
plt.tight_layout()
plt.show()

In [ ]:
# ── Application de SMOTE ──────────────────────────────────────────────────────
# SMOTE (Synthetic Minority Over-sampling Technique) :
# Pour chaque exemple minoritaire, trouve ses k plus proches voisins
# et génère de nouveaux exemples sur les segments les reliant.
# → Ce n'est pas une simple copie, mais une interpolation intelligente.

smote = SMOTE(random_state=42, k_neighbors=5)
X_tr_smote, y_tr_smote = smote.fit_resample(X_tr, y_tr)

print("📊 Distribution après SMOTE :")
unique_s, counts_s = np.unique(y_tr_smote, return_counts=True)
for u, c in zip(unique_s, counts_s):
    print(f"   Classe {u}: {c} ({c/len(y_tr_smote)*100:.1f}%)")
print(f"   Dataset agrandi : {len(X_tr)} → {len(X_tr_smote)} exemples")

In [ ]:
# ── Modèle simple pour la classification ─────────────────────────────────────
class FraudDetector(nn.Module):
    def __init__(self, input_dim=20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1)          # Sortie logit (sans Sigmoid → BCEWithLogitsLoss)
        )
    def forward(self, x):
        return self.net(x)


# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017) :
    FL(p) = -α * (1 - p)^γ * log(p)
    
    (1-p)^γ est le terme de 'focusing' : il réduit la contribution
    des exemples bien classés, forçant le modèle à se concentrer
    sur les cas difficiles (les fraudes rares et ambiguës).
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha  # Poids de la classe positive
        self.gamma = gamma  # Facteur de focusing

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = torch.exp(-bce)          # Probabilité de la bonne classe
        focal_weight = (1 - p_t) ** self.gamma
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * focal_weight * bce).mean()


def train_and_evaluate(X_train, y_train, X_test, y_test, loss_fn, label, epochs=30):
    """Entraîne et évalue un FraudDetector avec la loss donnée."""
    X_tr_t = torch.FloatTensor(X_train).to(DEVICE)
    y_tr_t = torch.FloatTensor(y_train).unsqueeze(1).to(DEVICE)
    X_te_t = torch.FloatTensor(X_test).to(DEVICE)
    y_te_t = torch.FloatTensor(y_test).unsqueeze(1).to(DEVICE)

    model = FraudDetector().to(DEVICE)
    opt   = Adam(model.parameters(), lr=1e-3)
    dataset = torch.utils.data.TensorDataset(X_tr_t, y_tr_t)
    loader  = DataLoader(dataset, batch_size=256, shuffle=True)

    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model(X_te_t)) > 0.5).cpu().numpy().flatten()
    f1 = f1_score(y_test, preds)
    print(f"\n── {label} ──")
    print(classification_report(y_test, preds, target_names=['Normal', 'Fraude'], digits=3))
    return f1


# ── Configuration des loss functions ─────────────────────────────────────────
# 1. BCE classique (baseline)
bce_plain = nn.BCEWithLogitsLoss()

# 2. BCE pondérée : poids inversement proportionnels aux fréquences
pos_weight = torch.tensor([y_tr.tolist().count(0) / y_tr.tolist().count(1)]).to(DEVICE)
bce_weighted = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# 3. Focal Loss
focal = FocalLoss(alpha=0.25, gamma=2.0).to(DEVICE)

print(f"⚖️  Poids de la classe positive (weighted BCE) : {pos_weight.item():.1f}×")

# ── Comparaison des approches ─────────────────────────────────────────────────
f1_scores = {}

f1_scores['BCE (baseline)']        = train_and_evaluate(X_tr, y_tr, X_te, y_te, bce_plain,    'BCE classique (pas de correction)')
f1_scores['BCE pondérée']          = train_and_evaluate(X_tr, y_tr, X_te, y_te, bce_weighted, 'BCE pondérée')
f1_scores['SMOTE + BCE']           = train_and_evaluate(X_tr_smote, y_tr_smote, X_te, y_te, bce_plain,  'SMOTE + BCE classique')
f1_scores['SMOTE + Focal Loss']    = train_and_evaluate(X_tr_smote, y_tr_smote, X_te, y_te, focal,      'SMOTE + Focal Loss')

In [ ]:
# ── Comparaison des F1-scores ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['tomato', 'orange', 'steelblue', 'royalblue']
bars = ax.bar(f1_scores.keys(), f1_scores.values(), color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, f1_scores.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
ax.set_title('Exercice 4 — F1-score (classe Fraude) selon la stratégie', fontsize=13)
ax.set_ylabel('F1-score')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### 📝 Analyse — Données déséquilibrées

Avec une BCE classique, le modèle apprend simplement à **tout prédire comme 'Normal'**, atteignant 98% d'accuracy mais un F1 nul sur la fraude — c'est le piège classique des métriques sur données déséquilibrées. L'accuracy est une métrique trompeuse ici ; F1, précision et rappel sont beaucoup plus révélateurs.

**SMOTE** est particulièrement efficace car il génère des exemples synthétiques *réalistes* par interpolation entre exemples voisins, plutôt que de simplement dupliquer. Cela enrichit l'espace de décision autour de la classe minoritaire sans créer d'overfitting.

La **Focal Loss** est complémentaire à SMOTE : là où SMOTE rééquilibre les données, la Focal Loss rééquilibre la dynamique d'apprentissage en forçant le modèle à se concentrer sur les exemples difficiles. Leur combinaison produit généralement les meilleurs résultats.

Ces techniques sont critiques dans des domaines à fort enjeu : détection de fraude, diagnostic médical (cancer rare), cybersécurité. Une mauvaise gestion du déséquilibre peut avoir des conséquences graves, comme rater 99% des fraudes.

---
# 🛡️ Exercice 5 — Robustesse et Entraînement Adversarial

## Objectif
Rendre un modèle résistant aux **attaques adversariales** — des perturbations quasi-invisibles qui trompent les réseaux de neurones.

### Qu'est-ce qu'une attaque adversariale ?
En ajoutant un bruit **calculé** (et non aléatoire) à une image, on peut faire dire n'importe quoi à un CNN — même si la perturbation est imperceptible à l'œil humain.

### FGSM (Fast Gradient Sign Method)
L'attaque la plus simple (Goodfellow et al., 2014) :
```
x_adv = x + ε × sign(∇_x Loss(θ, x, y))
```
On pousse l'image dans la direction qui **maximise la loss** (l'opposé de ce que fait l'optimiseur).

In [ ]:
# ── Chargement MNIST pour l'exercice 5 ────────────────────────────────────────
mnist_test_adv = torchvision.datasets.MNIST(
    root='./data', train=False, download=True,
    transform=transforms.ToTensor()  # Pas de normalisation : images dans [0,1]
)
test_loader_adv = DataLoader(mnist_test_adv, batch_size=256, shuffle=False)
print("✅ MNIST test chargé pour les attaques adversariales.")

In [ ]:
# ── CNN pour MNIST ────────────────────────────────────────────────────────────
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


def train_model(model, loader, epochs=5):
    """Entraînement standard (sans adversarial)."""
    model.train()
    opt = Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        total_loss = 0
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(X), y)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(loader):.4f}")
    return model


# Entraîner le modèle standard
print("🔵 Entraînement du modèle STANDARD (sans adversarial)...")
# Reuse mnist_loader from exercise 2 (same transform)
model_standard = MnistCNN().to(DEVICE)
model_standard = train_model(model_standard, mnist_loader, epochs=5)

In [ ]:
# ── Implémentation FGSM ───────────────────────────────────────────────────────
def fgsm_attack(model, images, labels, epsilon):
    """
    Génère des exemples adversariaux via FGSM.
    
    Args:
        model   : modèle cible
        images  : images originales, shape (B, C, H, W)
        labels  : vraies étiquettes
        epsilon : amplitude de la perturbation (ex: 0.1 = 10% de l'échelle [0,1])
    
    Returns:
        images_adv : images perturbées
    """
    images = images.clone().detach().requires_grad_(True)  # Activer le gradient sur les images
    outputs = model(images)
    loss = F.cross_entropy(outputs, labels)
    model.zero_grad()
    loss.backward()  # Calcul de ∇_x Loss

    # Perturbation = ε × signe du gradient
    perturbation = epsilon * images.grad.data.sign()
    images_adv = images.detach() + perturbation

    # Clamp pour garder les pixels dans [0, 1]
    return torch.clamp(images_adv, 0, 1)


def evaluate_accuracy(model, loader, epsilon=0.0, attack=False):
    """Évalue l'accuracy sur images propres ou adversariales."""
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if attack:
            images = fgsm_attack(model, images, labels, epsilon)
        with torch.no_grad():
            outputs = model(images)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)
    return 100. * correct / total


# Évaluation sur différents epsilon
epsilons = [0.0, 0.05, 0.1, 0.2, 0.3]
accs_standard = []

print("📊 Accuracy du modèle STANDARD sous attaque FGSM :")
for eps in epsilons:
    acc = evaluate_accuracy(model_standard, test_loader_adv, eps, attack=(eps > 0))
    accs_standard.append(acc)
    print(f"   ε={eps:.2f} → Accuracy: {acc:.1f}%")

In [ ]:
# ── Entraînement Adversarial ──────────────────────────────────────────────────
# Principe : à chaque batch, générer des exemples adversariaux
# et les inclure dans l'entraînement (mélangé avec les vraies images).
# Le modèle apprend à être correct sur les deux types d'entrées.

def adversarial_train(model, loader, epochs=5, epsilon=0.1):
    model.train()
    opt  = Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)

            # ── Passe sur images propres ──────────────────────────────────────
            opt.zero_grad()
            loss_clean = crit(model(X), y)

            # ── Passe sur images adversariales ────────────────────────────────
            X_adv = fgsm_attack(model, X, y, epsilon)
            loss_adv = crit(model(X_adv), y)

            # Loss totale = moyenne des deux
            loss = 0.5 * loss_clean + 0.5 * loss_adv
            loss.backward()
            opt.step()
            total_loss += loss.item()

        print(f"  Epoch {epoch+1}/{epochs} | Loss adversariale: {total_loss/len(loader):.4f}")
    return model


print("🟢 Entraînement ADVERSARIAL (ε=0.1)...")
model_robust = MnistCNN().to(DEVICE)
model_robust = adversarial_train(model_robust, mnist_loader, epochs=5, epsilon=0.1)

# Évaluation du modèle robuste
accs_robust = []
print("\n📊 Accuracy du modèle ROBUSTE sous attaque FGSM :")
for eps in epsilons:
    acc = evaluate_accuracy(model_robust, test_loader_adv, eps, attack=(eps > 0))
    accs_robust.append(acc)
    print(f"   ε={eps:.2f} → Accuracy: {acc:.1f}%")

In [ ]:
# ── Visualisation des exemples adversariaux ───────────────────────────────────
model_standard.eval()
sample_images, sample_labels = next(iter(test_loader_adv))
sample_images, sample_labels = sample_images[:5].to(DEVICE), sample_labels[:5].to(DEVICE)
adv_images = fgsm_attack(model_standard, sample_images, sample_labels, epsilon=0.2)

with torch.no_grad():
    preds_clean = model_standard(sample_images).argmax(1).cpu()
    preds_adv   = model_standard(adv_images).argmax(1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    axes[0, i].imshow(sample_images[i, 0].cpu().numpy(), cmap='gray')
    axes[0, i].set_title(f'Original\nPrédit: {preds_clean[i].item()}', fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].imshow(adv_images[i, 0].detach().cpu().numpy(), cmap='gray')
    color = 'red' if preds_adv[i].item() != sample_labels[i].item() else 'green'
    axes[1, i].set_title(f'Adversarial\nPrédit: {preds_adv[i].item()}', fontsize=9, color=color)
    axes[1, i].axis('off')

plt.suptitle('FGSM (ε=0.2) — Images originales vs adversariales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Courbe de robustesse ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epsilons, accs_standard, 'o-', label='Standard',  color='tomato',    linewidth=2, markersize=8)
ax.plot(epsilons, accs_robust,   's-', label='Adversarial', color='royalblue', linewidth=2, markersize=8)
ax.set_title('Exercice 5 — Robustesse sous attaque FGSM', fontsize=13)
ax.set_xlabel('Epsilon (amplitude de la perturbation)')
ax.set_ylabel('Accuracy (%)')
ax.legend()
plt.tight_layout()
plt.show()

### 📝 Réflexion — Robustesse et Entraînement Adversarial

Les attaques adversariales révèlent une faiblesse fondamentale des réseaux de neurones : ils apprennent des **corrélations statistiques** plutôt qu'une compréhension sémantique robuste. Une perturbation de quelques pixels, imperceptible à l'œil humain mais maximisant la loss, peut tromper un modèle avec 99% de confiance — ce qui pose des problèmes critiques dans des applications comme la conduite autonome ou la reconnaissance médicale.

L'entraînement adversarial, bien qu'efficace, implique un **trade-off** inévitable : le modèle robuste est légèrement moins précis sur les images propres. C'est le prix de la robustesse — le modèle doit apprendre des représentations plus générales et moins spécialisées.

Le domaine de la **robustesse adversariale** est encore ouvert : des attaques plus sophistiquées (PGD, AutoAttack) contournent souvent les défenses FGSM. Les recherches actuelles explorent des certifications formelles de robustesse et des architectures intrinsèquement plus stables comme les réseaux à convolutions lipschitziennes.

---
# 🎓 Conclusion Générale

Ce notebook vous a conduit à travers **5 défis majeurs** du deep learning avancé :

| Exercice | Technique | Bénéfice clé |
|----------|-----------|-------------|
| **1 - Optimisation** | Cosine Annealing + Gradient Clipping | Convergence plus rapide et stable |
| **2 - GAN** | DCGAN sur MNIST | Génération de données synthétiques réalistes |
| **3 - RL** | DQN avec Experience Replay | Agent autonome apprenant par essai-erreur |
| **4 - Déséquilibre** | SMOTE + Focal Loss | Détection fiable de la classe minoritaire |
| **5 - Robustesse** | FGSM + Adversarial Training | Résistance aux attaques adversariales |

Ces compétences forment le socle du **ML avancé production-grade** et sont directement applicables à des problèmes réels de l'industrie.